# My Content · Google Colab con enlace del proxy

Ejecuta las celdas **1 a 6** en orden. La **celda 6** inicia My Content y muestra un botón y una URL para abrir la aplicación en otra pestaña, mediante el proxy de Colab. No necesita ngrok.

1. En **Entorno de ejecución → Cambiar tipo de entorno**, elige GPU.
2. En **Secretos** (icono de llave) guarda `GROQ_API_KEY`, `HF_TOKEN` y `CONTENTAPP_CLAVE` y permite el acceso desde esta libreta. También puedes introducirlos en campos ocultos.
3. Para varias voces, acepta las [condiciones de pyannote Community-1](https://huggingface.co/pyannote/speaker-diarization-community-1) con la misma cuenta del token.

Incluye Smart Split actualizado: voces por hablante, conservación de fondo con Demucs y formatos YouTube, TikTok y fondo difuminado. Instala los motores; la generación de video requiere elegir sus modelos en la celda **5B** o en el gestor de la app. No se descargan todos los modelos de video, ya que algunos exceden los recursos de Colab.

**Datos temporales:** `/content` se pierde al eliminar el entorno; descarga los resultados con la celda 8.

**Acceso:** el enlace depende de tu sesión de Google y del entorno activo. Google marca la apertura en otra pestaña como poco fiable por restricciones del navegador; si aparece 403 o una página vacía, usa la celda 6B para abrir dentro de Colab. [Implementación oficial del proxy](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py).

Colab restringe usar principalmente interfaces web en entornos gratuitos sin saldo de unidades. Para ese uso, consulta los [planes y límites oficiales](https://research.google.com/colaboratory/faq.html). La libreta no evita los límites ni mantiene artificialmente la sesión activa.

Para dejar trabajando el bot: ejecuta **6C** y después **6D. Mantener vigilancia**. Deja 6D ejecutándose; revisa el servidor cada 30 segundos. Para ejecutar otras celdas, detén primero 6D. Si se recupera un túnel con URL nueva, actualízala en tu PC. Esto no elimina los límites de sesión de Colab.


In [ ]:
# @title 1. Configuración
from pathlib import Path
import os, sys, subprocess, json, time, socket, signal, getpass, shutil, uuid, re

REPO_URL = "https://github.com/mateoperez08251-crypto/my-content.git"
REVISION = "main"  # O un commit concreto para repetir exactamente la misma versión.
ROOT = Path("/content/my-content-proxy")
APP = ROOT / "repo"
DATA = ROOT / "datos"
PORT = 5001
PREPARAR_DOBLAJE = True  # False: voz única, sin instalar VoxCPM2 ni el detector.
PREPARAR_FONDO = True    # Instala Demucs y descarga htdemucs para conservar el fondo.
ROOT.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

def run(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, **kwargs)

def stop_app():
    global app_process
    proc = globals().get("app_process")
    if proc is not None:
        # El grupo incluye los hijos (FFmpeg, voces, etc.), incluso si el padre terminó.
        try:
            os.killpg(proc.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            pass
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        proc.wait(timeout=10)
    app_process = None

if not 1024 <= PORT <= 65535:
    raise ValueError("El puerto debe estar entre 1024 y 65535.")
print("Carpeta de trabajo:", ROOT)
print("Espacio libre: %.1f GB" % (shutil.disk_usage(ROOT).free / 1024**3))
if shutil.which("nvidia-smi"):
    run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
else:
    print("Sin GPU. Selecciona GPU para doblaje local; Groq y la interfaz pueden funcionar con CPU.")


In [ ]:
# @title 2. Descargar o actualizar el repositorio
stop_app()

# Retirar solo la actualización incluida e intacta antes de volver a descargar Git.
import hashlib, json
_overlay_manifest = ROOT / 'my-content-overlay.json'
if (APP / '.git').is_dir() and _overlay_manifest.is_file():
    _managed = json.loads(_overlay_manifest.read_text(encoding='utf-8'))
    _allowed = ['bot/identidad.py', 'bot/db.py', 'bot/archivos.py', 'bot/motor.py', 'bot/remoto.py', 'bot/planificacion.py', 'bot/presets.py', 'smart_selection.py', 'smart_editor.py', 'content.py', 'templates/bot.html', 'gpu_video.py', 'smart_upscale.py', 'smart_upscale_worker.py', 'mejora_video.py', 'modulo_ia.py', 'render_progress.py', 'bot/humano.py', 'bot/plataformas/tiktok.py', 'calidad_video.py', 'yt_downloader.py', 'smart_moments.py', 'clips_virales.py', 'templates/index.html']
    for _name, _hash in _managed.items():
        _target = (APP / _name).resolve()
        if _name not in _allowed or not _target.is_relative_to(APP.resolve()):
            raise RuntimeError('Ruta de actualización no válida.')
        if not _target.exists():
            continue
        if hashlib.sha256(_target.read_bytes()).hexdigest() != _hash:
            raise RuntimeError('Hay cambios propios en ' + _name + '. Guárdalos antes de actualizar.')
    for _name in _managed:
        _target = (APP / _name).resolve()
        _tracked = subprocess.run(['git','-C',str(APP),'ls-files','--error-unmatch','--',_name], capture_output=True).returncode == 0
        if _tracked:
            run(['git','-C',str(APP),'restore','--worktree','--',_name])
        elif _target.is_file():
            _target.unlink()
    _overlay_manifest.unlink()
if not (APP / ".git").is_dir():
    if APP.exists() and any(APP.iterdir()):
        raise RuntimeError("La carpeta repo ya contiene archivos sin Git; consérvala con otro nombre antes de continuar.")
    run(["git", "clone", "--no-checkout", REPO_URL, APP])
else:
    dirty = subprocess.check_output(["git", "-C", str(APP), "status", "--porcelain"], text=True)
    if dirty.strip():
        raise RuntimeError("Hay cambios locales en repo. Guárdalos antes de actualizar; no se borrarán.")
run(["git", "-C", APP, "fetch", "origin", REVISION])
run(["git", "-C", APP, "checkout", "--detach", "FETCH_HEAD"])
commit = subprocess.check_output(["git", "-C", str(APP), "rev-parse", "HEAD"], text=True).strip()
required = ["content.py", "smart_editor.py", "smart_dubbing.py", "smart_speakers.py",
            "smart_speaker_worker.py", "tts_worker.py", "runpod/requirements-runpod.txt"]
for name in required:
    if not (APP / name).is_file():
        raise RuntimeError("Falta un archivo del programa: " + name)
if "video_format" not in (APP / "smart_editor.py").read_text(encoding="utf-8"):
    raise RuntimeError("Esta revisión no incluye los formatos nuevos de Smart Split. Usa main actualizado.")
print("Código descargado:", commit)


In [ ]:
# @title 2B. Aplicar mejoras incluidas: clips distintos y calidad de imagen
import base64, hashlib, io, json, zipfile
_actualizacion = (
    '...'
)
_bundle = base64.b64decode(_actualizacion)
if hashlib.sha256(_bundle).hexdigest() != 'efd6cdf7eb11c19f251d9f82fb3537e62bf2cef307f2e15bd92e52d6971e5cee':
    raise RuntimeError('La actualización incluida está incompleta.')
_manifest = {}
with zipfile.ZipFile(io.BytesIO(_bundle)) as _zip:
    for _name in _zip.namelist():
        _target = (APP / _name).resolve()
        if not _target.is_relative_to(APP.resolve()):
            raise RuntimeError('Ruta no permitida en la actualización.')
        _target.parent.mkdir(parents=True, exist_ok=True)
        _data = _zip.read(_name)
        _target.write_bytes(_data)
        _manifest[_name] = hashlib.sha256(_data).hexdigest()
(ROOT / 'my-content-overlay.json').write_text(json.dumps(_manifest), encoding='utf-8')
print('Mejoras instaladas: selección por humor, drama y temas, contexto antes del remate, aceptación automática y calidad de imagen.')


In [ ]:
# @title 3. Instalar aplicación, FFmpeg y separación del fondo
stop_app()
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libgl1", "libglib2.0-0",
     "libsndfile1", "fonts-noto-cjk", "build-essential"])
# uv proporciona Python 3.12 independiente del Python del kernel de Colab.
run([sys.executable, "-m", "pip", "install", "--quiet", "uv>=0.8,<1"])

def ensure_environment(folder):
    folder = Path(folder)
    binary = folder / "bin/python"
    if binary.exists():
        check = subprocess.run([str(binary), "-c",
            "import sys,pip; assert sys.version_info[:2] == (3,12); assert sys.prefix != sys.base_prefix"],
            capture_output=True, text=True)
        if check.returncode == 0:
            return str(binary)
    if folder.exists():
        backup = folder.with_name(folder.name + ".incompleto-" + uuid.uuid4().hex[:8])
        folder.rename(backup)
        print("Entorno incompleto conservado:", backup)
    run([sys.executable, "-m", "uv", "venv", "--python", "3.12", "--seed", folder])
    run([binary, "-m", "pip", "--version"])
    return str(binary)

PYTHON = ensure_environment(ROOT / "venv-app-312")
def pip_install(python, *packages):
    run([python, "-m", "pip", "install", "--disable-pip-version-check", "--no-cache-dir",
         "--retries", "3", "--timeout", "120", *packages])

packages = ["flask>=3,<4", "requests>=2.31", "pillow>=10.1,<12", "numpy>=1.26,<2",
            "imageio-ffmpeg>=0.5", "moviepy>=2,<3", "opencv-python-headless>=4.8,<4.12",
            "yt-dlp[default,deno]", "edge-tts>=7.2,<8", "huggingface_hub", "psutil", "pyperclip",
            "google-api-python-client", "google-auth-oauthlib", "soundfile"]
pip_install(PYTHON, *packages)
if PREPARAR_FONDO:
    # Demucs usa las API de audio anteriores a TorchCodec; se aísla de pyannote/VoxCPM2.
    index = "cu124" if shutil.which("nvidia-smi") else "cpu"
    pip_install(PYTHON, "torch==2.5.1", "torchaudio==2.5.1",
                "--index-url", "https://download.pytorch.org/whl/" + index)
    constraints = ROOT / "demucs-constraints.txt"
    constraints.write_text("torch==2.5.1\ntorchaudio==2.5.1\nnumpy<2\n", encoding="utf-8")
    pip_install(PYTHON, "demucs==4.0.1", "-c", str(constraints))
run([PYTHON, "-c", "import flask, moviepy, cv2, edge_tts, imageio_ffmpeg; print('Aplicación: dependencias OK')"])
print("Instalación de la aplicación terminada. Continúa con claves y modelos.")


## 4. Claves de la aplicación
La clave de **Groq** se usa para seleccionar clips, traducir y transcribir en nube. **HF_TOKEN** permite descargar el detector si ya aceptaste sus condiciones. **CONTENTAPP_CLAVE** es la contraseña que elijas para acceder a la app; el usuario puede ser `colab`.

No pegues claves en el código. Usa Secretos de Colab o las entradas ocultas de la siguiente celda. Tras cambiar claves o instalar modelos, vuelve a ejecutar la celda 6 para reiniciar la app.

In [ ]:
# @title 4. Preparar claves y directorios
def secret(name, prompt):
    value = os.environ.get(name, "")
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = ""
    return value or getpass.getpass(prompt)

# Conservar los motores configurados si se repite esta celda.
previous = globals().get("app_env", {})
app_env = os.environ.copy()
app_env.update({
    "CONTENTAPP_SERVIDOR": "1", "CONTENTAPP_DATA_DIR": str(DATA),
    "CONTENTAPP_PUERTO": str(PORT), "CONTENTAPP_VIDEO_PYTHON": previous.get("CONTENTAPP_VIDEO_PYTHON", PYTHON),
    "CONTENTAPP_AUTOBORRAR_MIN": "0", "CONTENTAPP_MOTOR_PERSISTENTE": "0",
    "HF_HOME": str(DATA / "huggingface"), "TORCH_HOME": str(DATA / "torch"),
    "HF_XET_CHUNK_CACHE_SIZE_BYTES": "0", "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8",
    "PYTHONUTF8": "1", "SMART_SPLIT_CJK_FONT": "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
})
if previous.get("SMART_DIARIZATION_PYTHON"):
    app_env["SMART_DIARIZATION_PYTHON"] = previous["SMART_DIARIZATION_PYTHON"]
app_env["CONTENTAPP_CLAVE"] = secret("CONTENTAPP_CLAVE", "Contraseña para abrir My Content: ").strip()
if not app_env["CONTENTAPP_CLAVE"]:
    raise ValueError("Escribe una contraseña y repite esta celda.")

# Solo pedir la clave de Groq, omitiendo HF_TOKEN de las solicitudes interactivas
for name, label in [("GROQ_API_KEY", "Clave de Groq (Enter para configurar después): ")]:
    value = secret(name, label).strip()
    if value:
        app_env[name] = value

print("Entorno y claves preparados; no se muestran sus valores.")

## 5. Instalar motores y descargar modelos
Con las opciones iniciales se preparan **Demucs htdemucs**, **VoxCPM2** y **pyannote Community-1**. La primera ejecución descarga varios GB y puede tardar. Se verifica la carga de cada modelo; los procesos terminan después para liberar memoria.

Si solo quieres voz única, cambia `PREPARAR_DOBLAJE = False` en la celda 1 y desmarca **Detectar hablantes** en Smart Split. Si omites Demucs, desmarca **Preservar fondo**. Los modelos de video y Whisper local son opcionales en la celda 5B.

In [ ]:
# @title 5. Preparar fondo, voces y detector de hablantes
stop_app()
if PREPARAR_FONDO:
    print("Descargando y verificando Demucs htdemucs…", flush=True)
    run([PYTHON, "-c", "from demucs.pretrained import get_model; model=get_model('htdemucs'); print('Demucs listo')"],
        cwd=APP, env=app_env)

def prepare_voice_runtime():
    if not shutil.which("nvidia-smi"):
        raise RuntimeError("Selecciona un entorno con GPU para instalar los motores locales.")
    info = subprocess.check_output(["nvidia-smi"], text=True)
    version = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", info)
    cuda = tuple(map(int, version.groups())) if version else (0, 0)
    if cuda < (12, 6):
        raise RuntimeError("Estos motores necesitan un controlador compatible con CUDA 12.6 o superior. Cambia de entorno GPU.")
    index = "cu128" if cuda >= (12, 8) else "cu126"
    constraints = ROOT / "motor-constraints.txt"
    constraints.write_text("torch==2.8.0\ntorchaudio==2.8.0\ntorchvision==0.23.0\ntorchcodec==0.7.0\n", encoding="utf-8")
    voice_python = ensure_environment(ROOT / "venv-voz-312")
    pip_install(voice_python, "torch==2.8.0", "torchaudio==2.8.0", "torchvision==0.23.0",
                "--index-url", "https://download.pytorch.org/whl/" + index)
    pip_install(voice_python, "-r", str(APP / "runpod/requirements-runpod.txt"),
                "soundfile", "librosa", "psutil", "-c", str(constraints))
    run([voice_python, "-c", "import torch; assert torch.cuda.is_available(), 'CUDA no disponible'; print(torch.ones(8, device='cuda').sum().item())"], env=app_env)
    app_env["CONTENTAPP_VIDEO_PYTHON"] = voice_python
    return voice_python, index, constraints

if PREPARAR_DOBLAJE:
    voice_python, torch_index, motor_constraints = prepare_voice_runtime()
    run([voice_python, APP / "tts_worker.py", "--instalar"], cwd=APP, env=app_env)
    print("Descargando y comprobando VoxCPM2…", flush=True)
    run([voice_python, "-c", """
import os
from pathlib import Path
from huggingface_hub import snapshot_download
from tts_worker import cargar_modelo
folder = Path(os.environ['CONTENTAPP_DATA_DIR']) / 'models' / 'video_ai' / 'voxcpm2'
snapshot_download('openbmb/VoxCPM2', local_dir=folder,
    allow_patterns=['config.json','model.safetensors','audiovae.pth','tokenizer.json',
                    'tokenizer_config.json','special_tokens_map.json','tokenization_voxcpm2.py'])
for name in ['config.json','model.safetensors','audiovae.pth','tokenizer.json']:
    if not (folder / name).is_file() or (folder / name).stat().st_size == 0:
        raise RuntimeError('Descarga incompleta: ' + name)
model = cargar_modelo(str(folder))
(folder / '.completo').write_text('VoxCPM2 cargado correctamente', encoding='utf-8')
print('VoxCPM2 listo')
"""], cwd=APP, env=app_env)
    if app_env.get("HF_TOKEN"):
        diar_python = ensure_environment(ROOT / "venv-diarizacion-312")
        pip_install(diar_python, "torch==2.8.0", "torchaudio==2.8.0",
                    "--index-url", "https://download.pytorch.org/whl/" + torch_index)
        pip_install(diar_python, "pyannote.audio==4.0.0", "torchcodec==0.7.0", "-c", str(motor_constraints))
        run([diar_python, "-c", """
import os
from pyannote.audio import Pipeline
pipeline = Pipeline.from_pretrained('pyannote/speaker-diarization-community-1', token=os.environ['HF_TOKEN'])
if pipeline is None:
    raise RuntimeError('Comprueba HF_TOKEN y acepta las condiciones del detector.')
print('Detector de hablantes listo')
"""], cwd=APP, env=app_env)
        app_env["SMART_DIARIZATION_PYTHON"] = diar_python
    else:
        print("HF_TOKEN omitido. No se instalará el detector de hablantes (pyannote).")
else:
    print("Varias voces omitidas. Desmarca Detectar hablantes en el modal antes de exportar.")
print("Preparación terminada. Puedes iniciar la app en la celda 6.")

In [ ]:
# @title 5B. Elegir modelos de mejora de imagen y otros motores
MODELO_MEJORA = "Real-ESRGAN x2" # @param ["Sin modelo IA", "Real-ESRGAN x2", "Real-ESRGAN x4"]
WHISPER_LOCAL = "No descargar" # @param ["No descargar", "Whisper Turbo", "Whisper Large V3"]
# Otros IDs del gestor, separados por comas. Ejemplo: rife47
OTROS_MODELOS = "" # @param {type:"string"}
# x2: recomendado para video y menor consumo. x4: más ampliación, no siempre más calidad.
# Esta celda descarga y selecciona el modelo; activa Mejorar imagen: IA en el preset de cada cuenta.
MODELOS_EXTRA = []
_mejoras = {"Real-ESRGAN x2": ("realesrgan_x2", "RealESRGAN_x2plus.pth"),
            "Real-ESRGAN x4": ("realesrgan_x4", "RealESRGAN_x4plus.pth")}
_whisper = {"Whisper Turbo": "whisper_large_v3_turbo", "Whisper Large V3": "whisper_large_v3"}
if MODELO_MEJORA in _mejoras:
    MODELOS_EXTRA.append(_mejoras[MODELO_MEJORA][0])
if WHISPER_LOCAL in _whisper:
    MODELOS_EXTRA.append(_whisper[WHISPER_LOCAL])
MODELOS_EXTRA.extend(x.strip() for x in OTROS_MODELOS.split(',') if x.strip())
MODELOS_EXTRA = list(dict.fromkeys(MODELOS_EXTRA))
if MODELOS_EXTRA:
    stop_app()
    voice_python = app_env.get('CONTENTAPP_VIDEO_PYTHON')
    if not voice_python:
        voice_python, _, _ = prepare_voice_runtime()
    run([voice_python, APP / 'runpod/descargar_modelo.py', *MODELOS_EXTRA], cwd=APP, env=app_env)
    if MODELO_MEJORA in _mejoras:
        model = DATA / 'models' / 'video_ai' / _mejoras[MODELO_MEJORA][1]
        app_env['CONTENTAPP_ESRGAN_MODEL'] = str(model)
        run([voice_python, '-c', "import sys, torch; from mejora_video import cargar_esrgan; assert torch.cuda.is_available(), 'CUDA no disponible'; cargar_esrgan(sys.argv[1], torch, escala=int(sys.argv[2])); print('Modelo de mejora cargado en GPU correctamente')",
             model, '4' if MODELO_MEJORA.endswith('x4') else '2'], cwd=APP, env=app_env)
    else:
        app_env.pop('CONTENTAPP_ESRGAN_MODEL', None)
    print('Modelos descargados y comprobados. Continúa con la celda 6 y luego 6C.')
else:
    app_env.pop('CONTENTAPP_ESRGAN_MODEL', None)
    print('Sin descargas adicionales. Puedes usar calidad máxima sin IA.')

# Diagnóstico de GPU y exportación: la tarjeta de IA y el codificador son cosas distintas.
import shutil
if shutil.which('nvidia-smi'):
    run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], env=app_env)
run([PYTHON, '-c', "from gpu_video import hay_nvenc; print('Exportación por GPU (NVENC): disponible' if hay_nvenc() else 'Exportación por CPU: esta GPU o FFmpeg no ofrece NVENC')"], cwd=APP, env=app_env)


In [ ]:
# @title 6. Iniciar My Content y obtener URL para otra ventana
from google.colab import output
from IPython.display import display, HTML
import html, urllib.request, urllib.error, urllib.parse, base64

def redacted_log(filename="servidor.log"):
    path = ROOT / filename
    text = path.read_text(encoding="utf-8", errors="replace")[-12000:] if path.exists() else "Sin registro."
    for key in ("CONTENTAPP_CLAVE", "GROQ_API_KEY", "HF_TOKEN"):
        value = app_env.get(key)
        if value:
            text = text.replace(value, "[OCULTO]")
    return text

def show_proxy_link():
    if globals().get("app_process") is None or app_process.poll() is not None:
        raise RuntimeError("La app no está activa. Ejecuta la celda 6.")
    try:
        url = output.eval_js(f"google.colab.kernel.proxyPort({int(PORT)}, {{cache: false}})", timeout_sec=30)
        parsed = urllib.parse.urlparse(url or "")
        if parsed.scheme != "https" or not parsed.netloc or parsed.username or parsed.password:
            raise RuntimeError("Colab no devolvió una URL HTTPS válida.")
        display(HTML('<a href="' + html.escape(url, quote=True) +
            '" target="_blank" rel="noopener noreferrer" style="display:inline-block;padding:14px 22px;background:#166534;color:white;border-radius:8px;text-decoration:none;font-weight:600">Abrir My Content en otra ventana</a>'))
        print("URL:", url)
        print("Si pide acceso: usuario colab y la contraseña de la celda 4.")
        print("Mantén la libreta abierta y el entorno conectado. Si aparece 403, utiliza la celda 6B.")
        return url
    except Exception as exc:
        print("La app está activa, pero no se pudo obtener el enlace del proxy:", type(exc).__name__)
        print("Usa la celda 6B para ver la app dentro de Colab.")
        return None

def wait_for_app(proc, timeout=90):
    deadline = time.monotonic() + timeout
    credentials = base64.b64encode(("colab:" + app_env["CONTENTAPP_CLAVE"]).encode()).decode()
    while time.monotonic() < deadline:
        if proc.poll() is not None:
            return False
        try:
            req = urllib.request.Request(f"http://127.0.0.1:{PORT}/api/health",
                headers={"Authorization": "Basic " + credentials, "X-Forwarded-For": "127.0.0.1"})
            with urllib.request.urlopen(req, timeout=2) as response:
                if response.status == 200 and json.load(response).get("ok") is True:
                    return True
        except (OSError, ValueError, urllib.error.URLError):
            pass
        time.sleep(1)
    return False

stop_app()
if not Path(PYTHON).is_file() or not app_env.get("CONTENTAPP_CLAVE"):
    raise RuntimeError("Completa las celdas de instalación y claves primero.")
# Sincroniza el puerto aunque se haya cambiado después de configurar las claves.
app_env["CONTENTAPP_PUERTO"] = str(PORT)
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError(f"El puerto {PORT} está ocupado. Cambia PORT y repite esta celda.")
log_path = ROOT / "servidor.log"
with log_path.open("w", encoding="utf-8") as log:
    app_process = subprocess.Popen([PYTHON, str(APP / "content.py"), "--servidor"],
        cwd=APP, env=app_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
if not wait_for_app(app_process):
    stop_app()
    print(redacted_log())
    raise RuntimeError("La app no arrancó. Revisa el registro anterior y la celda 7.")
print("My Content está activa.")
proxy_url = show_proxy_link()


In [ ]:
# @title 6B. Alternativa: abrir dentro de Colab
if globals().get("app_process") is None or app_process.poll() is not None:
    raise RuntimeError("Primero ejecuta la celda 6.")
from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=900, cache_in_notebook=False)


## Usar Smart Split
- Sube el video desde el selector del navegador; las rutas corresponden a Colab, no a tu PC.
- Elige idioma, detecta y escucha los hablantes y asigna las voces. Conserva el fondo si preparaste Demucs.
- Elige el formato: **TikTok/Shorts 9:16**, **YouTube 16:9** o **video completo centrado con fondo difuminado**.
- Para empezar, exporta un clip corto en calidad rápida. La separación de voces y efectos puede ser imperfecta.
- La interfaz es la misma que en escritorio, ejecutada como servidor. Los diálogos nativos de Windows y la bandeja no se utilizan. Conectar/publicar en redes requiere sus credenciales y configuración OAuth; el enlace temporal no configura esas cuentas.

Si falta memoria GPU, reduce el trabajo o descarga el modelo de la GPU desde la app. El enlace no convierte una sesión de Colab en alojamiento permanente.

In [ ]:
# @title 6C. Conectar el bot de tu PC (URL HTTPS con Cloudflare)
# Ejecuta después de 6. No requiere cuenta ni token de Cloudflare.
# La contraseña de My Content sigue siendo obligatoria.
import hashlib, platform
import urllib.request, urllib.error, urllib.parse, base64

if globals().get("app_process") is None or app_process.poll() is not None:
    raise RuntimeError("Primero inicia My Content con la celda 6.")
if not app_env.get("CONTENTAPP_CLAVE"):
    raise RuntimeError("Configura la contraseña en la celda 4.")
# Cierra el túnel de una ejecución anterior antes de reemplazar su función de cierre.
if callable(globals().get("stop_bot_tunnel")):
    stop_bot_tunnel()

bot_tunnel_process = None
bot_tunnel_url = None

def stop_bot_tunnel():
    global bot_tunnel_process, bot_tunnel_url
    proc = globals().get("bot_tunnel_process")
    if proc is not None:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=8)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait(timeout=5)
        else:
            proc.wait(timeout=5)
    bot_tunnel_process = None
    bot_tunnel_url = None

def install_cloudflared():
    version = "2026.10.3"
    arch = {"x86_64": "amd64", "amd64": "amd64", "aarch64": "arm64", "arm64": "arm64"}.get(platform.machine().lower())
    if platform.system() != "Linux" or not arch:
        raise RuntimeError("Esta celda necesita un entorno Linux de Colab (amd64 o arm64).")
    checksums = {
        "amd64": "053b80b0a23031a038fdbf3c073f59f82b8f1f152aa67971d7383ffafc0dbab0",
        "arm64": "89297ebfb9e8c99c0dc013a1140601453cf3798d8306c5591651f4a3f5210be8",
    }
    binary = ROOT / f"cloudflared-{version}-linux-{arch}"

    def verified(path):
        if not path.is_file():
            return False
        digest = hashlib.sha256()
        with path.open("rb") as data:
            for block in iter(lambda: data.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest() == checksums[arch]

    if not verified(binary):
        url = f"https://github.com/cloudflare/cloudflared/releases/download/{version}/cloudflared-linux-{arch}"
        temporary = binary.with_suffix(".part")
        print("Descargando cloudflared desde Cloudflare…", flush=True)
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "My-Content-Colab"})
            with urllib.request.urlopen(request, timeout=30) as response, temporary.open("wb") as data:
                shutil.copyfileobj(response, data)
            if not verified(temporary):
                raise RuntimeError("La descarga de cloudflared está incompleta o no coincide con la versión oficial. Repite 6C.")
            temporary.chmod(0o700)
            os.replace(temporary, binary)
        finally:
            temporary.unlink(missing_ok=True)
    binary.chmod(0o700)
    return binary

def check_bot_url(url):
    # No reenviar la contraseña si una respuesta intenta redirigir a otra página.
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None

    opener = urllib.request.build_opener(NoRedirect())
    request = urllib.request.Request(url + "/api/health")
    try:
        with opener.open(request, timeout=10):
            raise RuntimeError("La app no exige contraseña. No se habilita la conexión.")
    except urllib.error.HTTPError as exc:
        if exc.code != 401:
            raise
    credentials = base64.b64encode(("colab:" + app_env["CONTENTAPP_CLAVE"]).encode()).decode()
    request = urllib.request.Request(url + "/api/health", headers={"Authorization": "Basic " + credentials})
    try:
        with opener.open(request, timeout=10) as response:
            try:
                payload = json.load(response)
            except ValueError:
                raise RuntimeError("El enlace no devolvió una respuesta válida de My Content.") from None
            if response.status != 200 or not isinstance(payload, dict) or payload.get("ok") is not True:
                raise RuntimeError("El servidor no confirmó que esté listo.")
    except urllib.error.HTTPError as exc:
        if exc.code in (401, 403):
            raise RuntimeError("La contraseña no coincide. Repite la celda 4, luego 6 y 6C.") from None
        raise

def wait_for_bot_tunnel(proc, log_path, timeout=120):
    deadline = time.monotonic() + timeout
    last_error = "Cloudflare todavía no publicó la URL"
    url = None
    while time.monotonic() < deadline:
        if proc.poll() is not None:
            raise RuntimeError("cloudflared se cerró. Revisa su registro en la celda 7 y repite 6C.")
        if url is None:
            text = log_path.read_text(encoding="utf-8", errors="replace")[-12000:]
            match = re.search(r'https://[a-z0-9](?:[a-z0-9-]*[a-z0-9])?\.trycloudflare\.com(?=[\s|"<>]|$)', text)
            if match:
                url = match.group(0)
        if url:
            try:
                check_bot_url(url)
                return url
            except urllib.error.HTTPError as exc:
                last_error = f"HTTP {exc.code}"
            except (OSError, urllib.error.URLError) as exc:
                last_error = type(exc).__name__
        time.sleep(2)
    raise RuntimeError(f"Cloudflare no quedó listo ({last_error}). Revisa la celda 7 y repite 6C.")

def start_bot_tunnel():
    global bot_tunnel_process, bot_tunnel_url
    stop_bot_tunnel()
    try:
        binary = install_cloudflared()
        # Configuración propia: no altera ni reutiliza túneles de otras aplicaciones.
        config = ROOT / "cloudflared-bot-empty.yml"
        config.write_text("{}\n", encoding="utf-8")
        log_path = ROOT / "cloudflared-bot.log"
        with log_path.open("w", encoding="utf-8") as log:
            bot_tunnel_process = subprocess.Popen([
                str(binary), "tunnel", "--config", str(config), "--no-autoupdate",
                "--protocol", "http2", "--url", f"http://127.0.0.1:{PORT}",
            ], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        print("Esperando a que Cloudflare conecte con My Content…", flush=True)
        bot_tunnel_url = wait_for_bot_tunnel(bot_tunnel_process, log_path)
    except BaseException:
        # También limpia si se interrumpe la celda durante el arranque.
        stop_bot_tunnel()
        raise
    print("URL para Servidor de cortes del bot:", bot_tunnel_url)
    print("En tu PC: pega esta URL y la contraseña de la celda 4, pulsa Probar.")
    print("Si reinicias la celda 6, ejecuta otra vez 6C y copia la URL nueva.")
    print("Espera a que los clips figuren guardados en tu PC antes de cerrar Colab.")
    return bot_tunnel_url

start_bot_tunnel()


In [ ]:
# @title 6D. Mantener vigilancia (dejar esta celda ejecutándose)
VIGILAR = True # @param {type:"boolean"}
# Revisa servicios cada 30 s. No amplía los límites de duración de Google Colab.

def supervision_tick(state, now):
    global app_process, proxy_url
    proc = globals().get("app_process")
    if proc is None or proc.poll() is not None:
        if now < state["app_retry"]:
            return
        print("El servidor se cerró. Intentando recuperarlo…", flush=True)
        # Limpia también los hijos del servidor anterior antes de reemplazarlo.
        stop_app()
        with (ROOT / "servidor.log").open("a", encoding="utf-8") as log:
            app_process = subprocess.Popen([PYTHON, str(APP / "content.py"), "--servidor"],
                cwd=APP, env=app_env, stdout=log, stderr=subprocess.STDOUT,
                start_new_session=True)
        if not wait_for_app(app_process):
            state["failures"] += 1
            state["app_retry"] = time.monotonic() + (60 if state["failures"] < 3 else 600)
            print("El servidor aún no inicia; se reintentará con pausa.", flush=True)
            return
        state["failures"] = 0
        print("Servidor recuperado.", flush=True)
        proxy_url = show_proxy_link()
    # Un servidor vivo puede estar renderizando: no se mata por responder lento.
    tunnel = globals().get("bot_tunnel_process")
    if state["watch_tunnel"] and (tunnel is None or tunnel.poll() is not None):
        if now < state["tunnel_retry"]:
            return
        state["tunnel_retry"] = time.monotonic() + 600
        print("Recuperando el enlace del bot…", flush=True)
        start_bot_tunnel()
        print("ATENCIÓN: Cloudflare puede cambiar la URL. Copia la URL mostrada en tu PC.", flush=True)

def vigilar_colab():
    proc = globals().get("app_process")
    if proc is None or proc.poll() is not None:
        raise RuntimeError("Ejecuta primero la celda 6 y, para el bot de tu PC, la 6C.")
    state = {"app_retry": 0, "tunnel_retry": 0, "failures": 0,
             "watch_tunnel": globals().get("bot_tunnel_process") is not None}
    last_status = float("-inf")
    print("Vigilancia activa cada 30 segundos. Deja esta celda ejecutándose.", flush=True)
    print("Detener esta celda detiene la vigilancia; el servidor sigue activo.", flush=True)
    print("Colab mantiene sus límites de sesión y consumo de unidades.", flush=True)
    try:
        while VIGILAR:
            now = time.monotonic()
            try:
                supervision_tick(state, now)
            except Exception as exc:
                # No imprimir mensajes de terceros que puedan incluir credenciales.
                state["app_retry"] = time.monotonic() + 60
                print("No se pudo completar la recuperación (" + type(exc).__name__ + "). Se reintentará.", flush=True)
            if now - last_status >= 900:
                proc = globals().get("app_process")
                alive = proc is not None and proc.poll() is None
                tunnel = globals().get("bot_tunnel_process")
                linked = tunnel is not None and tunnel.poll() is None
                print(time.strftime("[%H:%M] ") + "Servidor: " + ("activo" if alive else "en recuperación")
                      + " · Enlace del bot: " + ("activo" if linked else "sin conexión"), flush=True)
                last_status = now
            time.sleep(30)
    except KeyboardInterrupt:
        print("Vigilancia detenida. El servidor sigue activo; repite 6D para volver a vigilar.", flush=True)

if VIGILAR:
    vigilar_colab()
else:
    print("Vigilancia desactivada.")


In [ ]:
# @title 7. Ver diagnóstico y regenerar enlace
proc = globals().get("app_process")
print("Servidor:", "activo" if proc is not None and proc.poll() is None else "detenido")
print("Espacio libre: %.1f GB" % (shutil.disk_usage(ROOT).free / 1024**3))
if "redacted_log" in globals():
    print(redacted_log())
    if (ROOT / "cloudflared-bot.log").exists():
        print("Registro de Cloudflare:")
        print(redacted_log("cloudflared-bot.log"))
if proc is not None and proc.poll() is None:
    proxy_url = show_proxy_link()
else:
    print("Para iniciar o reiniciar, ejecuta la celda 6.")


In [ ]:
# @title 6. Iniciar My Content y obtener URL para otra ventana
from google.colab import output
from IPython.display import display, HTML
import html, urllib.request, urllib.error, urllib.parse, base64

def redacted_log():
    path = ROOT / "servidor.log"
    text = path.read_text(encoding="utf-8", errors="replace")[-12000:] if path.exists() else "Sin registro."
    for key in ("CONTENTAPP_CLAVE", "GROQ_API_KEY", "HF_TOKEN"):
        value = app_env.get(key)
        if value:
            text = text.replace(value, "[OCULTO]")
    return text

def show_proxy_link():
    if globals().get("app_process") is None or app_process.poll() is not None:
        raise RuntimeError("La app no está activa. Ejecuta la celda 6.")
    try:
        url = output.eval_js(f"google.colab.kernel.proxyPort({int(PORT)}, {{cache: false}})", timeout_sec=30)
        parsed = urllib.parse.urlparse(url or "")
        if parsed.scheme != "https" or not parsed.netloc or parsed.username or parsed.password:
            raise RuntimeError("Colab no devolvió una URL HTTPS válida.")
        display(HTML('<a href="' + html.escape(url, quote=True) +
            '" target="_blank" rel="noopener noreferrer" style="display:inline-block;padding:14px 22px;background:#166534;color:white;border-radius:8px;text-decoration:none;font-weight:600">Abrir My Content en otra ventana</a>'))
        print("URL:", url)
        print("Si pide acceso: usuario colab y la contraseña de la celda 4.")
        print("Mantén la libreta abierta y el entorno conectado. Si aparece 403, utiliza la celda 6B.")
        return url
    except Exception as exc:
        print("La app está activa, pero no se pudo obtener el enlace del proxy:", type(exc).__name__)
        print("Usa la celda 6B para ver la app dentro de Colab.")
        return None

def wait_for_app(proc, timeout=90):
    deadline = time.monotonic() + timeout
    credentials = base64.b64encode(("colab:" + app_env["CONTENTAPP_CLAVE"]).encode()).decode()
    while time.monotonic() < deadline:
        if proc.poll() is not None:
            return False
        try:
            req = urllib.request.Request(f"http://127.0.0.1:{PORT}/api/health",
                headers={"Authorization": "Basic " + credentials, "X-Forwarded-For": "127.0.0.1"})
            with urllib.request.urlopen(req, timeout=2) as response:
                if response.status == 200 and json.load(response).get("ok") is True:
                    return True
        except (OSError, ValueError, urllib.error.URLError):
            pass
        time.sleep(1)
    return False

stop_app()
if not Path(PYTHON).is_file() or not app_env.get("CONTENTAPP_CLAVE"):
    raise RuntimeError("Completa las celdas de instalación y claves primero.")
# Sincroniza el puerto aunque se haya cambiado después de configurar las claves.
app_env["CONTENTAPP_PUERTO"] = str(PORT)
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError(f"El puerto {PORT} está ocupado. Cambia PORT y repite esta celda.")
log_path = ROOT / "servidor.log"
with log_path.open("w", encoding="utf-8") as log:
    app_process = subprocess.Popen([PYTHON, str(APP / "content.py"), "--servidor"],
        cwd=APP, env=app_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
if not wait_for_app(app_process):
    stop_app()
    print(redacted_log())
    raise RuntimeError("La app no arrancó. Revisa el registro anterior y la celda 7.")
print("My Content está activa.")
proxy_url = show_proxy_link()


In [ ]:
# @title 8. Descargar resultados
from google.colab import files
import zipfile
RESULTADOS = [DATA / "videos_procesados", DATA / "salidas"]
archive = ROOT / "resultados_my_content.zip"
allowed = {".mp4", ".mov", ".webm", ".wav", ".mp3", ".srt", ".vtt", ".png", ".jpg", ".jpeg"}
media = []
for folder in RESULTADOS:
    if folder.is_dir() and not folder.is_symlink():
        for path in folder.rglob("*"):
            if path.is_file() and not path.is_symlink() and path.suffix.lower() in allowed:
                if path.resolve().is_relative_to(folder.resolve()):
                    media.append(path)
if not media:
    print("Todavía no hay resultados. Si elegiste otra carpeta, cambia RESULTADOS.")
else:
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as bundle:
        for path in media:
            bundle.write(path, path.relative_to(DATA))
    print(f"Preparados {len(media)} archivos. Para videos grandes usa el panel Archivos de Colab.")
    files.download(str(archive))


## Validación y referencias
Preparada el 17 de octubre de 2026 para [mateoperez08251-crypto/my-content](https://github.com/mateoperez08251-crypto/my-content). Se confirmó en GitHub la presencia de los formatos nuevos y el doblaje actualizado.

- [Proxy de Colab y limitación de pestañas nuevas](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py).
- [Límites de Colab](https://research.google.com/colaboratory/faq.html).
- [Versiones de PyTorch](https://pytorch.org/get-started/previous-versions/).
- [Requisitos de pyannote 4](https://github.com/pyannote/pyannote-audio/blob/4.0.0/pyproject.toml).
- [Compatibilidad de TorchCodec](https://github.com/meta-pytorch/torchcodec#compatibility-with-torch-versions).

Se comprueban estructura, sintaxis, lógica de arranque y generación del enlace fuera de Colab. Esta libreta no se ha ejecutado aquí en una sesión real de Colab: las descargas de modelos, la GPU y el acceso a la URL se verifican al ejecutar sus celdas. Si una celda falla, resuelve ese error antes de avanzar.